# NUTDTS 816 Time Series Analysis
## L16 Neural networks for sequences

Lab notebook for Chapter 8 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Set up the LSTM forecaster (Chapter 9, section 9.1); the evaluation follows in Lab 17

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, torch, time
import tsdata
torch.manual_seed(816); np.random.seed(816)
dem = tsdata.daily_demand()
L, H = 56, 14                       # input window (8 weeks) and output horizon (2 weeks)

def make_windows(z, start, end):
    """Supervised windows from a scaled array: inputs z[t-L:t], targets z[t:t+H], for t in [start+L, end-H]."""
    X, Y = [], []
    for t in range(start + L, end - H + 1):
        X.append(z[t - L:t]); Y.append(z[t:t + H])
    return torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1), torch.tensor(np.array(Y), dtype=torch.float32)

class LSTMForecaster(torch.nn.Module):
    def __init__(self, units=32, dropout=0.1):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size=1, hidden_size=units, batch_first=True)
        self.drop = torch.nn.Dropout(dropout); self.head = torch.nn.Linear(units, H)   # multi-output: all H steps at once
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)            # h_n: final hidden state, shape (1, batch, units)
        return self.head(self.drop(h_n[-1]))

def train_lstm(y_train, units=32, lr=1e-3, max_epochs=80, patience=8, batch=64, val_frac=0.15, verbose=False):
    """Scale on the training data only, split off a validation tail for early stopping, train with Adam."""
    y = y_train.values.astype('float32'); mu, sd = y.mean(), y.std(); z = (y - mu) / sd
    n_val = int(len(z) * val_frac)
    Xtr, Ytr = make_windows(z, 0, len(z) - n_val); Xva, Yva = make_windows(z, len(z) - n_val - L, len(z))
    model = LSTMForecaster(units); opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = torch.nn.MSELoss()
    best, bad, hist, best_state = np.inf, 0, [], None
    for ep in range(max_epochs):
        model.train(); perm = torch.randperm(len(Xtr)); tl = 0
        for i in range(0, len(Xtr), batch):
            idx = perm[i:i + batch]; opt.zero_grad(); loss = loss_fn(model(Xtr[idx]), Ytr[idx]); loss.backward(); opt.step(); tl += loss.item() * len(idx)
        model.eval()
        with torch.no_grad(): vl = loss_fn(model(Xva), Yva).item()
        hist.append((tl / len(Xtr), vl))
        if vl < best - 1e-4: best, bad, best_state = vl, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else: bad += 1
        if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, (mu, sd), pd.DataFrame(hist, columns=['train', 'val'])

def lstm_forecast(model, scaler, y_train):
    mu, sd = scaler; z = (y_train.values[-L:].astype('float32') - mu) / sd
    with torch.no_grad(): out = model(torch.tensor(z, dtype=torch.float32).view(1, L, 1)).numpy().ravel()
    return out * sd + mu

split = int(len(dem) * 0.8)
t0 = time.time(); model, scaler, hist = train_lstm(dem.iloc[:split], verbose=True)
print(f'Trained {sum(p.numel() for p in model.parameters()):,} parameters in {len(hist)} epochs, {time.time() - t0:.1f} s on CPU; best validation MSE (scaled) = {hist.val.min():.3f}')
ax = hist.plot(figsize=(7, 2.8), lw=1.5); ax.set_xlabel('epoch'); ax.set_ylabel('MSE (scaled units)'); ax.set_title('Training and validation loss with early stopping')
_caption = 'Validation loss stops improving after a few dozen epochs; training continues to fall, which is the beginning of over-fitting. Early stopping keeps the best validation epoch.'

In [ ]:
keras_code = '''
import tensorflow as tf
model = tf.keras.Sequential([tf.keras.layers.Input((L, 1)), tf.keras.layers.LSTM(32), tf.keras.layers.Dropout(0.1), tf.keras.layers.Dense(H)])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
model.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=80, batch_size=64, callbacks=[es], verbose=0)
'''

### Carried forward from Lab 15 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata
dem = tsdata.daily_demand()     # simulated daily demand (GWh), 2022-2025, weekly + annual seasonality, holidays, trend
NG_HOLIDAYS = [f'{y}-{md}' for y in range(2022, 2027) for md in ['01-01', '05-01', '06-12', '10-01', '12-25', '12-26']]

def make_features(y, horizon, lags=(1, 2, 3, 7, 14, 21, 28, 364), windows=(7, 28, 91), keep_unlabelled=False):
    """Feature table for a DIRECT h-step forecaster: row t uses information up to t, target is y[t + horizon].
    keep_unlabelled=True keeps the final rows whose target is not yet observed (used to build the forecast row)."""
    df = pd.DataFrame(index=y.index)
    for k in lags: df[f'lag{k}'] = y.shift(k)
    for w in windows:
        df[f'rmean{w}'] = y.shift(1).rolling(w).mean(); df[f'rstd{w}'] = y.shift(1).rolling(w).std()
    df['diff7'] = y.shift(1) - y.shift(8)
    tgt_idx = y.index + pd.Timedelta(days=horizon)         # calendar features describe the TARGET date (known in advance)
    df['dow'] = tgt_idx.dayofweek; df['month'] = tgt_idx.month; df['doy_sin'] = np.sin(2 * np.pi * tgt_idx.dayofyear / 365.25); df['doy_cos'] = np.cos(2 * np.pi * tgt_idx.dayofyear / 365.25)
    df['holiday'] = tgt_idx.strftime('%Y-%m-%d').isin(NG_HOLIDAYS).astype(int)
    df['target'] = y.shift(-horizon)
    feats = [c for c in df.columns if c != 'target']
    return df.dropna(subset=feats) if keep_unlabelled else df.dropna()

F7 = make_features(dem, horizon=7)
print(F7.shape); print(F7.iloc[[0, -1], :8].round(2).to_string())

In [ ]:
import lightgbm as lgb
# A trending series: can a tree model follow it?
rng = np.random.default_rng(7); t = np.arange(400); y_tr = pd.Series(50 + 0.2 * t + 5 * np.sin(2 * np.pi * t / 30) + rng.normal(0, 1.5, 400))
X = pd.DataFrame({'lag1': y_tr.shift(1), 'lag2': y_tr.shift(2), 'lag30': y_tr.shift(30), 'phase': np.sin(2 * np.pi * t / 30)}).dropna(); yy = y_tr.loc[X.index]
tr_idx = X.index < 300
m_level = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, verbose=-1).fit(X[tr_idx], yy[tr_idx])
m_diff  = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, verbose=-1).fit(X[tr_idx], (yy - X['lag1'])[tr_idx])
fig, ax = plt.subplots(figsize=(9, 3.2)); y_tr.plot(ax=ax, lw=1, label='series (trend + cycle)')
pd.Series(m_level.predict(X[~tr_idx]), index=X.index[~tr_idx]).plot(ax=ax, lw=2, color='#A0302A', label='LightGBM on the level: flattens at the training maximum')
pd.Series(X['lag1'][~tr_idx] + m_diff.predict(X[~tr_idx]), index=X.index[~tr_idx]).plot(ax=ax, lw=2, color='#B8860B', label='LightGBM on the difference: follows the trend')
ax.axvline(300, color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_title('Why the target must be made stationary for tree models')
_caption = 'Trained on the first 300 points, a boosted-tree model of the level cannot predict above the highest value it has seen; the same model trained on the one-step change has no such limit.'

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
H = 14
split = int(len(dem) * 0.8); origins = list(range(split, len(dem) - H, 21))    # an origin every three weeks
print(f'{len(origins)} origins from {dem.index[split].date()}, horizon {H} days')

def lgbm_direct_forecast(y_train, H, params=dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8, colsample_bytree=0.8, verbose=-1)):
    fc = []
    for h in range(1, H + 1):
        F = make_features(y_train, h); F['target_d'] = F['target'] - F['lag1']          # direct model of the CHANGE from the last observed value
        feats = [c for c in F.columns if c not in ('target', 'target_d')]
        m = lgb.LGBMRegressor(**params).fit(F[feats], F['target_d'])
        row = make_features(y_train, h, keep_unlabelled=True).loc[[y_train.index[-1]], feats]   # features at the origin, target h days ahead
        fc.append(y_train.iloc[-1] + m.predict(row)[0])
    return np.array(fc)

errs = {'LightGBM direct': [], 'ETS(A,Ad,A) weekly': [], 'Seasonal naive (7)': []}
for T in origins:
    ytr, act = dem.iloc[:T], dem.iloc[T:T + H].values
    errs['LightGBM direct'].append(act - lgbm_direct_forecast(ytr, H))
    ets = ETSModel(ytr, error='add', trend='add', damped_trend=True, seasonal='add', seasonal_periods=7, initialization_method='estimated').fit(disp=False)
    errs['ETS(A,Ad,A) weekly'].append(act - ets.forecast(H).values)
    errs['Seasonal naive (7)'].append(act - np.tile(ytr.iloc[-7:].values, 2)[:H])
q = np.mean(np.abs(dem.values[7:] - dem.values[:-7]))
mase = pd.DataFrame({k: np.mean(np.abs(np.array(v)), axis=0) / q for k, v in errs.items()}, index=[f'h={k}' for k in range(1, H + 1)])
print(mase.round(3).iloc[[0, 1, 2, 6, 13]].to_string()); print('\nAverage MASE over 14 horizons:'); print(mase.mean().round(3).to_string())

In [ ]:
ax = mase.plot(figsize=(8.5, 3.4), marker='o', ms=3, lw=1.5); ax.set_xlabel('horizon (days)'); ax.set_ylabel('MASE (scaled by weekly naive)'); ax.set_title('Daily demand: rolling-origin accuracy by horizon'); ax.legend(fontsize=8)
_caption = 'A split decision: the weekly ETS is more accurate for the first few days, the direct LightGBM forecaster from about a week out, and both beat the seasonal naive throughout.'

In [ ]:
# Which features matter? Gain-based importance from the 7-day-ahead model
F = make_features(dem.iloc[:split], 7); F['target_d'] = F['target'] - F['lag1']; feats = [c for c in F.columns if c not in ('target', 'target_d')]
m7 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15, verbose=-1).fit(F[feats], F['target_d'])
imp = pd.Series(m7.booster_.feature_importance('gain'), index=feats).sort_values(ascending=False)
print((100 * imp / imp.sum()).round(1).head(10).to_string())

## Neural networks for sequences

## Exercises

5. Explain to a product manager why a model trained on 500 products' sales histories can forecast a product launched last month, and what assumption that relies on.
6. For an LSTM layer with 16 units on a univariate input, compute the number of parameters. How many 30-day windows can be made from three years of daily data? Is the ratio comfortable?
7. Describe, in three sentences each, what the forget gate, the input gate and the output gate do, using the example of a demand series with a one-off strike month.

In [ ]:
# Your work here
